# Bollinger Band Reversal (BBR)

## Import Libs

In [3]:
import pandas as pd 
import numpy as np
import os
from pandas import DataFrame, Series
import plotly.graph_objects as go

## Functions

### Get FE Data

In [623]:
def get_fe_price_data(
        filename: str = "FE_V2_GBPUSD_15mins_1yr_End_20250311.csv"
        ) -> DataFrame:
    """
    Return the FE price data as a DatetimeIndexed 
    DataFrame set to US/Eastern TZ
    """
    FOLDER = "price_data"
    PATH = f"{os.getcwd()}/{FOLDER}"
    df = pd.read_csv(f"{PATH}/{filename}")
    df["Date"] = pd.DatetimeIndex(df["Date"], tz="US/Eastern")
    df.set_index("Date", inplace=True)
    return df

In [ ]:
# Set new columns 
gains_cols = [ 
    "Win", "Loss", 
    "TP", "SL", 
    "Gain",
    "Trade_Start", "Trade_End"
    ]

### Simulate Short Positions (Range-Based)

In [188]:
def range_short_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        range_type: str = "ADR"
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = high.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = high.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out

        for i in range(len(sl_window)): 
            if sl_window.iloc[i] >= (df["Close"] + df[range_type] * sl_pct_range ):
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD
        # Take profit price
        tp = df["Close"] - (df[range_type] * tp_pct_range)
        
        # trade window
        tp_window = low[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["Close"] - tp
        sl_pips = df["Close"] - (df["Close"] + df[range_type] * sl_pct_range) 
        if tp_window.min() <= tp:
            win = 1
            gain = df["Close"] - tp
        else:
            loss = 1
            if stop is True:
                gain = sl_pips
            else:
                gain = df["Close"] - close_window.iloc[-1] if close_window.empty is False else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end
    
    return data

# Set new columns 
# gains_cols = [ 
#     "Win", "Loss", 
#     "TP", "SL", 
#     "Gain",
#     "Trade_Start", "Trade_End"
#     ]

#df[pct_50_idr_cols] = df.apply(get_bear_bbr_pip_gain, axis=1, args=[df["High"], df["Low"], df["Close"]], result_type='expand')


### Simulate Long Positions (Range-Based)

In [189]:
def range_long_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        range_type: str = "ADR"
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = low.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = low.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out

        for i in range(len(sl_window)): 
            if sl_window.iloc[i] <= (df["Close"] - df[range_type] * sl_pct_range ):
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD

        # Take profit price
        tp = df["Close"] + (df[range_type] * tp_pct_range)
           
        # trade window
        tp_window = high[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = tp - df["Close"]
        sl_pips = (df["Close"] - df[range_type] * sl_pct_range) - df["Close"] 
        if tp_window.max() >= tp:
            win = 1
            gain = tp - df["Close"]
        else:
            loss = 1
            if stop is True:
                gain = sl_pips
            else:
                gain = close_window.iloc[-1] - df["Close"] if close_window.empty is False else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data

# Set new columns 
# long_gains_cols = [
#     "Win", "Loss", 
#     "TP", "SL", 
#     "Gain",
#     "Trade_Start", "Trade_End",
#     ]

#df[pct_50_idr_cols] = df.apply(get_bull_bbr_pip_gain, axis=1, args=[df["High"], df["Low"], df["Close"]], result_type='expand')


### Range-Based Gains (Long/Short)

In [1015]:
def get_range_signal_gains(df: DataFrame, long: str, short: str, sl_pct_r, tp_pct_r):
    long_df = df.copy() 
    short_df = df.copy()
    # long
    long_df[gains_cols] = long_df.apply(
        range_long_gains,
        axis=1,
        args=[df["High"],df["Low"],df["Close"], 
            long, sl_pct_r, tp_pct_r],
        result_type='expand',
        range_type="ADR"
    )
    # short
    short_df[gains_cols] = short_df.apply(
        range_short_gains,
        axis=1,
        args=[df["High"],df["Low"],df["Close"], 
            short, sl_pct_r, tp_pct_r],
        result_type='expand',
        range_type="ADR"
    )
    return pd.concat([long_df, short_df])

### Simulate Long/Short Trend Position

In [955]:
def trend_gains(
        df: Series, 
        close: Series,
        sma: Series,
        long_signal: str,
        short_signal: str
        ):
    """Get the pip gain and apply to df
    
    - long_signal: name of buy signal
    - short_signal: name of sell signal
    """

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None

    if df[long_signal] is True or df[short_signal] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = close.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        sma_window = sma.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        
        # find exit timestamp:
        for i in range(len(close_window)): 
            # exit condition
            if df[long_signal] is True:
                exit_condition = 1 if close_window.iloc[i] < sma_window.iloc[i] else 0
            elif df[short_signal] is True:
                exit_condition = 1 if close_window.iloc[i] > sma_window.iloc[i] else 0
            # get stop loss time:
            if exit_condition == 1:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = close_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD if (START+TD) < END else START

        # Win / Loss / Gain / Pips
        if df[long_signal] is True:
            gain = close[sl_ts] - df["Close"]
        elif df[short_signal] is True:
            gain = df["Close"] - close[sl_ts]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["ATR4"]
        sl_pips = -(df["ATR4"])
        win = 1 if gain > 0 else 0
        loss = 1 if gain < 0 else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data



### Trade Stats

In [943]:
def trade_stats(df: DataFrame, signal_name: str):
    win_count = df.query(f"{signal_name} == True and Win > 0")[f"{signal_name}"].count()
    loss_count = df.query(f"{signal_name} == True and Loss > 0")[f"{signal_name}"].count()
    total_trades = win_count + loss_count
    win_rate = win_count/total_trades * 100
    win = df.query(f"{signal_name} == True and Gain > 0")["Gain"]
    loss = df.query(f"{signal_name} == True and Gain < 0")["Gain"]
    win_avg_pips = win.mean()
    loss_avg_pips = loss.mean()
    win_pips = win.sum()
    loss_pips = loss.sum()
    total_pips = win_pips + loss_pips

    stats = {
        "Symbol": df["Symbol"].iloc[0],
        "Start": df.index.min(),
        "End": df.index.max(),
        "Win_Count": win_count,
        "Loss_Count": loss_count,
        "Total_Trades": total_trades,
        "Win_Rate": win_rate,
        "Avg_Win": win_avg_pips,
        "Avg_Loss": loss_avg_pips,
        "Win_Pips": win_pips,
        "Loss_Pips": loss_pips,
        "Total_Pips": total_pips
    }

    return stats

def signal_stats(df: DataFrame, signals: list["str"]):
    stats = [trade_stats(df, signal) for signal in signals]
    return stats

def get_trend_signal_stats(df: DataFrame, long: str, short: str, sma: str = "SMA4"):
    # Apply gains to df
    df[gains_cols] = df.apply(
        trend_gains,
        axis=1,
        args=[df["Close"], df[sma], long, short],
        result_type='expand'
    )

    # get stats for long / short signals
    gain_stats = signal_stats(df, [long,short])

    # build stats dataframe
    return pd.DataFrame(data=gain_stats,
                index=[long,short]
                )

## Price Data Files 

In [1035]:
price_data_files = os.listdir(f"{os.getcwd()}/price_data")

## Breakout

In [999]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
signals_df = get_fe_price_data(filename=path)

gain_stats_df = get_trend_signal_stats(signals_df, "BBU_BO", "BBL_BO")
gain_stats_df



/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")


,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
BBU_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,167,315,482,34.647303,0.001191,-0.000723,0.198855,-0.227665,-0.028810
BBL_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,170,291,461,36.876356,0.001229,-0.000664,0.208915,-0.193270,0.015645


### Results (first 20)

In [555]:
signals_df[gains_cols].query("Win > 0 or Loss > 0").iloc[0:20]

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End
Date,,,,,,,
2024-03-13 03:15:00-04:00,0.0,1.0,0.000806,-0.000806,-0.001020,2024-03-13 03:30:00-04:00,2024-03-13 04:00:00-04:00
2024-03-13 23:30:00-04:00,0.0,1.0,0.000467,-0.000467,-0.000135,2024-03-13 23:45:00-04:00,2024-03-14 00:45:00-04:00
2024-03-14 03:00:00-04:00,1.0,0.0,0.000471,-0.000471,0.001365,2024-03-14 03:15:00-04:00,2024-03-14 06:30:00-04:00
2024-03-14 04:00:00-04:00,1.0,0.0,0.000696,-0.000696,0.000315,2024-03-14 04:15:00-04:00,2024-03-14 06:30:00-04:00
2024-03-14 08:45:00-04:00,1.0,0.0,0.001711,-0.001711,0.004440,2024-03-14 09:00:00-04:00,2024-03-14 12:00:00-04:00
2024-03-14 09:15:00-04:00,1.0,0.0,0.001802,-0.001802,0.002955,2024-03-14 09:30:00-04:00,2024-03-14 12:00:00-04:00
2024-03-14 20:00:00-04:00,0.0,1.0,0.000471,-0.000471,-0.000160,2024-03-14 20:15:00-04:00,2024-03-14 21:15:00-04:00
2024-03-14 20:15:00-04:00,0.0,1.0,0.000546,-0.000546,-0.000525,2024-03-14 20:30:00-04:00,2024-03-14 21:15:00-04:00
2024-03-15 05:30:00-04:00,1.0,0.0,0.000781,-0.000781,0.000105,2024-03-15 05:45:00-04:00,2024-03-15 07:00:00-04:00


## SMA Breakout

In [642]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
signals_df = get_fe_price_data(filename=path)

gain_stats_df = get_trend_signal_stats(signals_df, "Bull_SMA_BO", "Bear_SMA_BO")
gain_stats_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")


,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_SMA_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,62,72,134,46.268657,0.001673,-0.000812,0.103695,-0.058435,0.045260
Bear_SMA_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,51,100,151,33.774834,0.001822,-0.000826,0.092900,-0.082615,0.010285


### Results (first 20)

In [643]:
signals_df[gains_cols].query("Win > 0 or Loss > 0").iloc[0:20]

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End
Date,,,,,,,
2024-03-13 14:45:00-04:00,0.0,1.0,0.000566,-0.000566,-0.000370,2024-03-13 15:00:00-04:00,2024-03-13 16:00:00-04:00
2024-03-13 20:00:00-04:00,0.0,1.0,0.000385,-0.000385,-0.000275,2024-03-13 20:15:00-04:00,2024-03-13 20:45:00-04:00
2024-03-15 03:45:00-04:00,0.0,1.0,0.000676,-0.000676,-0.000635,2024-03-15 04:00:00-04:00,2024-03-15 04:15:00-04:00
2024-03-15 08:30:00-04:00,0.0,1.0,0.000829,-0.000829,-0.001055,2024-03-15 08:45:00-04:00,2024-03-15 09:00:00-04:00
2024-03-17 23:30:00-04:00,0.0,1.0,0.000300,-0.000300,-0.000070,2024-03-17 23:45:00-04:00,2024-03-18 01:15:00-04:00
2024-03-18 01:15:00-04:00,1.0,0.0,0.000276,-0.000276,0.000570,2024-03-18 01:30:00-04:00,2024-03-18 05:15:00-04:00
2024-03-18 08:30:00-04:00,1.0,0.0,0.000606,-0.000606,0.000085,2024-03-18 08:45:00-04:00,2024-03-18 11:45:00-04:00
2024-03-18 23:30:00-04:00,1.0,0.0,0.000535,-0.000535,0.003945,2024-03-18 23:45:00-04:00,2024-03-19 06:15:00-04:00
2024-03-20 00:45:00-04:00,0.0,1.0,0.000341,-0.000341,-0.000275,2024-03-20 01:00:00-04:00,2024-03-20 01:30:00-04:00


## Bollinger Band Reversals

In [644]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bull_bbr_df = get_fe_price_data(filename=path)
bear_bbr_df = get_fe_price_data(filename=path)

long_signal = "Bull_BBR_V2"
short_signal = "Bear_BBR_V2"

# long
bull_bbr_df[gains_cols] = bull_bbr_df.apply(
    range_long_gains,
    axis=1,
    args=[bull_bbr_df["High"],bull_bbr_df["Low"],bull_bbr_df["Close"], 
          long_signal, 1.5, 2.5],
    result_type='expand',
    range_type="ATR4"
)
# short
bear_bbr_df[gains_cols] = bear_bbr_df.apply(
    range_short_gains,
    axis=1,
    args=[bear_bbr_df["High"],bear_bbr_df["Low"],bear_bbr_df["Close"], 
          short_signal, 1.5, 2.5],
    result_type='expand',
    range_type="ATR4"
)

bull_bbr_gains_df = trade_stats(bull_bbr_df, long_signal)
bear_bbr_gains_df = trade_stats(bear_bbr_df, short_signal)

bbr_gains_df = pd.DataFrame(data=[bull_bbr_gains_df, bear_bbr_gains_df],
             index=[long_signal,short_signal]
             )

bbr_gains_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")


,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_BBR_V2,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,183,309,492,37.195122,0.002349,-0.001330,0.544886,-0.344341,0.200544
Bear_BBR_V2,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,150,354,504,29.761905,0.002351,-0.001313,0.458520,-0.404528,0.053992


In [645]:
bull_bbr_df[[*gains_cols, "Bull_BBR_C1","Bull_BBR_C2","Bull_BBR_C3","Bull_BBR_C4"]].nlargest(10, "Gain")

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,Bull_BBR_C1,Bull_BBR_C2,Bull_BBR_C3,Bull_BBR_C4
Date,,,,,,,,,,,
2025-02-06 07:45:00-05:00,1.0,0.0,0.007922,-0.004753,0.007922,2025-02-06 08:00:00-05:00,2025-02-06 16:45:00-05:00,NaN,NaN,NaN,True
2024-08-05 02:15:00-04:00,1.0,0.0,0.007772,-0.004663,0.007772,2024-08-05 02:30:00-04:00,2024-08-05 16:45:00-04:00,NaN,NaN,True,NaN
2025-01-20 20:00:00-05:00,0.0,1.0,0.009031,-0.005419,0.007675,2025-01-20 20:15:00-05:00,2025-01-21 16:45:00-05:00,NaN,NaN,True,NaN
2025-02-06 07:30:00-05:00,1.0,0.0,0.007637,-0.004582,0.007637,2025-02-06 07:45:00-05:00,2025-02-06 16:45:00-05:00,NaN,NaN,NaN,True
2025-02-12 09:00:00-05:00,1.0,0.0,0.007403,-0.004442,0.007403,2025-02-12 09:15:00-05:00,2025-02-12 16:45:00-05:00,NaN,NaN,True,NaN
2025-02-12 08:45:00-05:00,1.0,0.0,0.007147,-0.004288,0.007147,2025-02-12 09:00:00-05:00,2025-02-12 16:45:00-05:00,NaN,NaN,True,NaN
2024-08-05 02:00:00-04:00,1.0,0.0,0.006412,-0.003847,0.006412,2024-08-05 02:15:00-04:00,2024-08-05 16:45:00-04:00,NaN,NaN,True,NaN
2024-11-25 19:15:00-05:00,1.0,0.0,0.005909,-0.003546,0.005909,2024-11-25 19:30:00-05:00,2024-11-26 16:45:00-05:00,True,NaN,NaN,True
2025-01-15 02:15:00-05:00,1.0,0.0,0.005906,-0.003544,0.005906,2025-01-15 02:30:00-05:00,2025-01-15 16:45:00-05:00,True,NaN,NaN,NaN


## Breakout Momentum

In [646]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bom_df = get_fe_price_data(filename=path)

long_signal = "Bull_BM"
short_signal = "Bear_BM"

bom_gains_df = get_range_signal_gains(bom_df, long_signal, short_signal, 1, 1.25)
bom_stats = signal_stats(bom_gains_df, [long_signal,short_signal])
bom_stats_df = pd.DataFrame(data=bom_stats, index=[long_signal,short_signal])
bom_stats_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")


,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_BM,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,558,722,1280,43.593750,0.001134,-0.001006,0.691480,-0.672167,0.019312
Bear_BM,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,431,580,1011,42.631058,0.001303,-0.001032,0.599605,-0.566340,0.033265


### Results (first 20)

In [647]:
bom_gains_df
bom_gains_df[gains_cols].query("Win > 0 or Loss > 0").iloc[0:20]

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End
Date,,,,,,,
2024-03-14 03:15:00-04:00,1.0,0.0,0.000678,-0.000542,0.000678,2024-03-14 03:30:00-04:00,2024-03-14 08:15:00-04:00
2024-03-14 03:30:00-04:00,1.0,0.0,0.000823,-0.000659,0.000823,2024-03-14 03:45:00-04:00,2024-03-14 08:15:00-04:00
2024-03-14 03:45:00-04:00,1.0,0.0,0.000859,-0.000687,0.000859,2024-03-14 04:00:00-04:00,2024-03-14 08:15:00-04:00
2024-03-14 04:00:00-04:00,1.0,0.0,0.000870,-0.000696,0.000870,2024-03-14 04:15:00-04:00,2024-03-14 08:15:00-04:00
2024-03-14 04:15:00-04:00,0.0,1.0,0.001028,-0.000822,-0.000822,2024-03-14 04:30:00-04:00,2024-03-14 04:45:00-04:00
2024-03-14 04:30:00-04:00,1.0,0.0,0.000992,-0.000794,0.000992,2024-03-14 04:45:00-04:00,2024-03-14 08:15:00-04:00
2024-03-15 05:45:00-04:00,0.0,1.0,0.001000,-0.000800,-0.000800,2024-03-15 06:00:00-04:00,2024-03-15 08:30:00-04:00
2024-03-15 06:00:00-04:00,0.0,1.0,0.001031,-0.000825,-0.000825,2024-03-15 06:15:00-04:00,2024-03-15 08:30:00-04:00
2024-03-15 06:15:00-04:00,0.0,1.0,0.000923,-0.000739,-0.000739,2024-03-15 06:30:00-04:00,2024-03-15 07:00:00-04:00


# GBP

In [1037]:
gbp_data = [x for x in price_data_files if "GBP" in x]
eur_data = [x for x in price_data_files if "EUR" in x]
cad_data = [x for x in price_data_files if "CAD" in x]
jpy_data = [x for x in price_data_files if "JPY" in x]
aud_data = [x for x in price_data_files if "AUD" in x]
nzd_data = [x for x in price_data_files if "NZD" in x]
chf_data = [x for x in price_data_files if "CHF" in x]


In [1042]:
def range_based_multi_year_stats(
        df_list: list[DataFrame], 
        long: str, 
        short: str, 
        sl_pct_r: float, 
        tp_pct_r: float
        ):
    signal_stats_list = []
    gains_df_list = []
    for df in df_list:
        signal_gains_df = get_range_signal_gains(df, long, short, sl_pct_r, tp_pct_r)
        gains_df_list.append(signal_gains_df)
        long_short_stats = signal_stats(signal_gains_df.between_time("02:00","11:00"), [long,short])
        stats_df = pd.DataFrame(data=long_short_stats, index=[long,short])
        signal_stats_list.append(stats_df)
    return pd.concat(signal_stats_list), pd.concat(gains_df_list)

pd.options.display.max_rows = 100
gbp_data.sort()
gbp_data
gbp_df_list = [get_fe_price_data(filename=x) for x in gbp_data]
# eur_df_list = [get_fe_price_data(filename=x) for x in eur_data]
# cad_df_list = [get_fe_price_data(filename=x) for x in cad_data]
# jpy_df_list = [get_fe_price_data(filename=x) for x in jpy_data]
# aud_df_list = [get_fe_price_data(filename=x) for x in aud_data]
# nzd_df_list = [get_fe_price_data(filename=x) for x in nzd_data]
# chf_df_list = [get_fe_price_data(filename=x) for x in chf_data]

long_signal = "Bull_TM"
short_signal = "Bear_TM"

stats, gains = range_based_multi_year_stats(gbp_df_list, long_signal, short_signal, 0.04, 0.08)

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (48,49,60,62,63,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (46,48,51,56,61,63,65,67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (46,47,48,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory

In [1043]:
stats

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_TM,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,537,700,1237,43.411479,0.001046,-0.000683,0.642143,-0.425794,0.216349
Bear_TM,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,553,690,1243,44.489139,0.000953,-0.000681,0.593965,-0.422522,0.171443
Bull_TM,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,520,724,1244,41.800643,0.001371,-0.000903,0.769395,-0.616162,0.153233
Bear_TM,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,559,811,1370,40.802920,0.001314,-0.000824,0.821135,-0.613306,0.207829
Bull_TM,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,535,714,1249,42.834267,0.001062,-0.000661,0.652297,-0.417044,0.235254
Bear_TM,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,551,705,1256,43.869427,0.000886,-0.000711,0.536165,-0.458194,0.077971
Bull_TM,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,512,707,1219,42.001641,0.000846,-0.000752,0.483012,-0.487361,-0.004349
Bear_TM,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,442,636,1078,41.001855,0.001097,-0.000697,0.570638,-0.389170,0.181467
Bull_TM,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,514,724,1238,41.518578,0.000913,-0.000664,0.532518,-0.435029,0.097489
Bear_TM,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,526,676,1202,43.760399,0.001265,-0.000627,0.751625,-0.381414,0.370211


In [1077]:
g = gains.sort_index()
x = g['2026-03-04 17:15:00-05:00':'2026-03-05 19:30:00-05:00'][[*gains_cols, "ATR4", "ADR"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00")
x.query("Gain > 0")["Gain"].sum() + x.query("Gain < 0")["Gain"].sum()
x

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,ATR4,ADR
Date,,,,,,,,,
2026-03-05 02:00:00-05:00,1.0,0.0,0.000811,-0.000405,0.000811,2026-03-05 02:15:00-05:00,2026-03-05 03:00:00-05:00,0.001294,0.010132
2026-03-05 02:15:00-05:00,1.0,0.0,0.000811,-0.000405,0.000811,2026-03-05 02:30:00-05:00,2026-03-05 02:45:00-05:00,0.001238,0.010132
2026-03-05 02:30:00-05:00,0.0,1.0,0.000811,-0.000405,-0.000405,2026-03-05 02:45:00-05:00,2026-03-05 02:45:00-05:00,0.001406,0.010132
2026-03-05 03:15:00-05:00,0.0,1.0,0.000811,-0.000405,-0.000405,2026-03-05 03:30:00-05:00,2026-03-05 03:30:00-05:00,0.001756,0.010132
2026-03-05 04:00:00-05:00,0.0,1.0,0.000811,-0.000405,-0.000405,2026-03-05 04:15:00-05:00,2026-03-05 04:15:00-05:00,0.002286,0.010132
2026-03-05 06:45:00-05:00,1.0,0.0,0.000811,-0.000405,0.000811,2026-03-05 07:00:00-05:00,2026-03-05 07:00:00-05:00,0.001242,0.010132
2026-03-05 08:30:00-05:00,1.0,0.0,0.000811,-0.000405,0.000811,2026-03-05 08:45:00-05:00,2026-03-05 09:00:00-05:00,0.001256,0.010132
2026-03-05 09:30:00-05:00,0.0,1.0,0.000811,-0.000405,-0.000405,2026-03-05 09:45:00-05:00,2026-03-05 09:45:00-05:00,0.001431,0.010132
2026-03-05 10:15:00-05:00,1.0,0.0,0.000811,-0.000405,0.000811,2026-03-05 10:30:00-05:00,2026-03-05 12:30:00-05:00,0.001880,0.010132


In [1023]:
def trend_based_multi_year_stats(
        df_list: list[DataFrame], 
        long: str, 
        short: str,
        sma: str
        ):
    signal_stats_list = []
    for df in df_list:
        stats_df = get_trend_signal_stats(df, long, short, sma)
        signal_stats_list.append(stats_df)
    return pd.concat(signal_stats_list)

gbp_df_list = [get_fe_price_data(filename=x) for x in gbp_data]
long_signal = "Bull_SMA_BO"
short_signal = "Bear_SMA_BO"
trend_based_multi_year_stats(gbp_df_list, long_signal, short_signal, "SMA4")

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (46,47,48,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (34,45,46,48,49,50,51,56,57,58,59,60,61,62,63,65,66,67,68,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (48,49,60,62,63,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_2283/1547086338.py:10: DtypeWarning: Columns (46,48,51,56,61,63,65,67,68) have mixed types. Specify dtype option on import or set low_memory

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_SMA_BO,GBPUSD,2025-03-11 17:15:00-04:00,2026-03-11 16:45:00-04:00,37,81,118,31.355932,0.001079,-0.000836,0.039940,-0.067695,-0.027755
Bear_SMA_BO,GBPUSD,2025-03-11 17:15:00-04:00,2026-03-11 16:45:00-04:00,65,87,152,42.763158,0.001307,-0.000592,0.084975,-0.051520,0.033455
Bull_SMA_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,62,72,134,46.268657,0.001410,-0.000623,0.087405,-0.044855,0.042550
Bear_SMA_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,57,95,152,37.500000,0.001299,-0.000639,0.074015,-0.060665,0.013350
Bull_SMA_BO,GBPUSD,2023-03-13 17:15:00-04:00,2024-03-12 16:45:00-04:00,56,80,136,41.176471,0.001268,-0.000769,0.071025,-0.061540,0.009485
Bear_SMA_BO,GBPUSD,2023-03-13 17:15:00-04:00,2024-03-12 16:45:00-04:00,63,110,173,36.416185,0.001091,-0.000722,0.068735,-0.079365,-0.010630
Bull_SMA_BO,GBPUSD,2021-03-11 17:15:00-05:00,2022-03-11 16:45:00-05:00,50,79,129,38.759690,0.001025,-0.000758,0.051270,-0.059890,-0.008620
Bear_SMA_BO,GBPUSD,2021-03-11 17:15:00-05:00,2022-03-11 16:45:00-05:00,66,125,191,34.554974,0.001198,-0.000701,0.079070,-0.087665,-0.008595
Bull_SMA_BO,GBPUSD,2022-03-10 17:15:00-05:00,2023-03-10 16:45:00-05:00,43,95,138,31.159420,0.001617,-0.001127,0.069545,-0.107035,-0.037490
Bear_SMA_BO,GBPUSD,2022-03-10 17:15:00-05:00,2023-03-10 16:45:00-05:00,52,108,160,32.500000,0.001764,-0.001333,0.091715,-0.144005,-0.052290
